In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# LeNet

The preceding sections supplied the parts: the convolutional layer
(that section), padding and stride (that section),
channels (that section), and pooling (that section). None
of them has classified anything yet. We now assemble them into a complete
classifier and ask the obvious question: is it any better than the MLP?

The number to beat is that of the MLP of that section,
about 87% validation accuracy on Fashion-MNIST after 30 epochs. Recall that
the MLP reads an image as 784 numbers in no particular order
(that section), whereas convolutional layers keep the spatial
layout and use the same kernel at every position. To keep the comparison
fair, we train the convolutional classifier with the same loss, optimizer,
and number of epochs as the MLP and compare their validation accuracy. One
number will not settle the matter, however. We therefore also look at what
the first layer has learned, and we repeat the comparison on images whose
pixels have been shuffled.

The network we build is *LeNet*, after Yann LeCun, who developed it with his
colleagues at AT&T Bell Labs to recognize handwritten digits
[@LeCun.Bottou.Bengio.ea.1998]. It was the result of a decade of work
that began with an early convolutional network for images trained end to
end by backpropagation [@LeCun.Boser.Denker.ea.1989]. LeNet-5, the
version of 1998, reached an error rate below 1% on handwritten digits, on
par with the support vector machines of the time, and variants of it read
the handwritten amounts on bank checks in commercial systems. It showed that
a network trained from examples could replace a hand-engineered recognition
pipeline.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## The LeNet Architecture

LeNet consists of two parts, shown in the figure: a convolutional
encoder with two convolutional layers, and a dense block with three fully
connected layers.

![Data flow in LeNet on one $28\times28$ Fashion-MNIST image, with the feature maps drawn to scale. Each shaded window produces the dark unit it points to in the next layer. The three fully connected layers have 120, 84, and 10 outputs.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/lenet.svg)

Each convolutional block consists of a convolutional layer with a
$5\times5$ kernel, a sigmoid activation, and $2\times2$ average pooling with
stride 2. The first convolutional layer has 6 output channels and the second
has 16.

The encoder's output has shape (batch size, height, width, channels).

The dense block needs one vector per example, so we first flatten the
encoder's output into a matrix with one row per example. Its three fully
connected layers have 120, 84, and 10 outputs. The last layer has one output
per class, as in softmax regression.

This is the common teaching version of LeNet-5, not an exact reproduction.
The network of 1998 had the same alternating convolutions and pooling and
the same widths of 6, 16, 120, and 84. It differed in
details that need not concern us: a scaled tanh as activation, pooling with
a trainable weight and bias, partial connections between the two
convolutional layers, and radial-basis output units. One difference does
leave a trace in our code. The original took $32\times32$ images, so our
first convolutional layer pads its $28\times28$ input by two pixels on each
side, which gives it the same extent and the same $28\times28$ output.

In a framework, LeNet is a `Sequential` container of these layers. We use
Xavier initialization (that section) for the weights and zero
biases.

NNX layers draw their weights in the constructor. We pass Xavier's uniform
initializer to each layer; the biases start at zero, the Flax default.

In [ ]:
class LeNet(d2l.Classifier):
    """The layout of LeNet-5, with sigmoid activations and average pooling."""
    def __init__(self, lr=0.1, num_classes=10, kernel_init=None, rngs=None):
        super().__init__()
        self.save_hyperparameters(ignore=['rngs', 'kernel_init'])
        rngs = nnx.Rngs(d2l.get_key()) if rngs is None else rngs
        kernel_init = (nnx.initializers.xavier_uniform() if kernel_init is None
                       else kernel_init)

        def avg_pool(x):  # 2x2 windows, stride 2
            return nnx.avg_pool(x, window_shape=(2, 2), strides=(2, 2))

        def flatten(x):
            return x.reshape((x.shape[0], -1))

        self.net = nnx.Sequential(
            nnx.Conv(1, 6, kernel_size=(5, 5), padding=2,
                     kernel_init=kernel_init, rngs=rngs), nnx.sigmoid,
            avg_pool,
            nnx.Conv(6, 16, kernel_size=(5, 5), padding='VALID',
                     kernel_init=kernel_init, rngs=rngs), nnx.sigmoid,
            avg_pool,
            flatten,
            nnx.Linear(400, 120, kernel_init=kernel_init, rngs=rngs),
            nnx.sigmoid,
            nnx.Linear(120, 84, kernel_init=kernel_init, rngs=rngs),
            nnx.sigmoid,
            nnx.Linear(84, num_classes, kernel_init=kernel_init, rngs=rngs))

## Layer Shapes

A wrong padding or a missing flattening step changes the shape of every
layer that follows it, so we check the shapes before training.
the figure lists the layers in compressed notation, with the
shape of the tensor that passes between them. Does the code build this
network? To find out, we pass a single-channel $28 \times 28$ image through
it and print the output shape after each layer. The method `layer_summary`
does this for any classifier whose layers sit in a sequential container
`net`; the next chapter uses it for larger networks.

![Compressed notation for LeNet. Beside each arrow is the shape of the tensor it carries, as channels@height×width.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/lenet-vert.svg)

In [ ]:
@d2l.add_to_class(d2l.Classifier)
def layer_summary(self, X_shape):
    X = jnp.zeros(X_shape)
    for layer in self.net.layers:
        X = layer(X)
        # Activations and pooling are plain functions: print their names
        name = getattr(layer, '__name__', type(layer).__name__)
        print(f'{name:<22} {tuple(X.shape)}')

model = LeNet()
model.layer_summary((1, 28, 28, 1))

It does, and the shapes follow the rules of that section. A
$5\times5$ kernel without padding shrinks the height and the width by 4. The
first convolutional layer, with its padding of 2, keeps $28\times28$; the
second has none and turns $14\times14$ into $10\times10$. Each pooling
layer halves the height and the width, from 28 to 14 and from 10 to 5.
Meanwhile the number of channels grows from 1 to 6 to 16, so the encoder
trades spatial resolution for channels. The flattened output has $16 \times
5 \times 5 = 400$ entries, which the dense block reduces to one output per
class.

## Parameters and Multiply-Adds

With the shapes in hand we can say what every layer costs. Recall from
that section that a convolutional layer with
$c_\textrm{i}$ input channels, $c_\textrm{o}$ output channels, and a
$k\times k$ kernel has $c_\textrm{i} c_\textrm{o} k^2$ weights and
$c_\textrm{o}$ biases, and that it applies each weight once at every output
position. A fully connected layer uses each of its weights only once per
example. the table applies these counts to LeNet.

<!-- tbl-caption:Parameters and multiply-adds per example of the layers of LeNet. Pooling and activation layers have no parameters. -->

| layer | output | parameters | multiply-adds |
|:--|:--|--:|--:|
| $5\times5$ convolution, 1 to 6 channels | 6@28×28 | $150 + 6 = 156$ | $150 \times 784 = 117{,}600$ |
| $5\times5$ convolution, 6 to 16 channels | 16@10×10 | $2{,}400 + 16 = 2{,}416$ | $2{,}400 \times 100 = 240{,}000$ |
| fully connected, 400 to 120 | 120 | $48{,}000 + 120 = 48{,}120$ | $48{,}000$ |
| fully connected, 120 to 84 | 84 | $10{,}080 + 84 = 10{,}164$ | $10{,}080$ |
| fully connected, 84 to 10 | 10 | $840 + 10 = 850$ | $840$ |
| total | | $61{,}706$ | $416{,}520$ |

The two kinds of layers divide the work in opposite ways. The convolutional
layers hold 4% of the parameters and perform 86% of the multiply-adds,
because each of their weights is used at every position of a feature map.
The first fully connected layer holds 78% of the parameters and performs 12%
of the multiply-adds. Compared with the MLP, which has 203,530 parameters and
performs about 203,000 multiply-adds per image, LeNet has fewer than a third
of the parameters and performs about twice the arithmetic.

## Training

As for the MLP, we minimize the cross-entropy by minibatch stochastic
gradient descent for 30 epochs. The minibatches hold 128 images, half the
MLP's 256. Unfortunately, the learning rate cannot stay where it was. The trouble is
the sigmoid, which passes back at most $1/4$ of the gradient
(that section). The gradient that reaches the
weights of the first convolutional layer has passed backward through four
sigmoids, one after each of the four hidden layers. At the MLP's learning
rate of 0.1, LeNet learns slowly: its validation accuracy stays at chance
for the first two to four epochs and reaches only about 73% after 10. We use
a learning rate of 2.0 instead. The factor of twenty is empirical: the
argument says only that the rate must grow.

As in that section, the `Trainer` runs on the GPU
if there is one.

In [ ]:
trainer = d2l.Trainer(max_epochs=30, num_gpus=1)
data = d2l.FashionMNIST(batch_size=128)
model = LeNet(lr=2.0)
trainer.fit(model, data)

After 30 epochs LeNet reaches a validation accuracy of about 88--90%,
against about 87% for the MLP.

The start of the curve deserves a second look. In some runs the accuracy
rises at once. In others it sits at chance, 10%, for a few epochs, rarely
more, with the loss stuck near $\ln 10 \approx 2.3$, and then takes
off. Nothing distinguishes these runs but their random initial weights.
While the loss stays at $\ln 10$ the network gives every image the same
ten probabilities: after four sigmoids, the part of its output that depends
on the input is still too small to matter. An exercise measures this, and
the plateau will return below.

### The Learned Features

For softmax regression the weights were ten templates, one blurred garment
per class (that section), and the hidden units of the MLP
weighed regions of the image (that section). Both could
be drawn as $28\times28$ pictures, because every weight belonged to one
pixel. A kernel of LeNet belongs to no pixel in particular. It is a
$5\times5$ array of weights, too small to tell the eye much, so we look at
what it does as well as at what it is. We take a sneaker and a shirt from
the validation set and draw the six kernels of the first convolutional layer
in the top row, and below each kernel the feature map that it produces for
the two garments, after the sigmoid. Every image has its own gray scale,
from its smallest value (black) to its largest (white).

In [ ]:
X, y = trainer.prepare_batch(next(iter(data.val_dataloader())))
labels = d2l.numpy(y)
X = d2l.stack([X[int(np.argmax(labels == c))] for c in (7, 6)], 0)

In [ ]:
layers = model.net.layers
maps = layers[1](layers[0](X))  # The first convolution and its sigmoid
# Channels-last to channels-first, so that maps[k, c] is one 28x28 map
images, maps = d2l.numpy(X[..., 0]), d2l.numpy(maps).transpose(0, 3, 1, 2)
kernels = d2l.numpy(layers[0].kernel[...])[:, :, 0].transpose(2, 0, 1)

In [ ]:
names = data.text_labels([7, 6])
axes = d2l.show_images(
    [kernels[0], *kernels, images[0], *maps[0], images[1], *maps[1]], 3, 7,
    titles=['', *[f'channel {c}' for c in range(1, 7)],
            names[0], *[''] * 6, names[1], *[''] * 6], cmap='gray')
axes[0].set_visible(False)  # No kernel above the input images

Two kinds of channels appear, and they appear in every run. Some reproduce
the silhouette of the garment, filled in, either bright on a dark background
or inverted. Such a channel reports, roughly, whether its neighborhood is
bright or dark. The others light up only along the outline, most of them on
its left or on its right side. They respond to vertical edges, and several
of their kernels show how: a column of bright weights beside a column of
dark ones, a wider relative of the kernel $(1, -1)$ of
that section. There we had to hand gradient descent the
edge map before it found the detector. Here nobody asked for edges. The
network was given garments and their labels, and found that edges help to
tell them apart. Which channel does what changes from run to run.

### Shuffled Pixels

How much should we read into LeNet's gain of one to three points over the
MLP? Not too much. The MLP's own accuracy moves by about a point from one
epoch to the next (that section), and LeNet differs from
the MLP in its depth, its activation function, and its learning rate as well
as in its convolutions. A sharper test removes the one thing that only the
convolutions know about, namely which pixels are neighbors. This is the
shuffling that the chapter introduction imagined: we fix one random order of
the 784 pixel positions and rearrange every image, for training and
validation alike, into that order.

In [ ]:
perm = np.random.permutation(28 * 28)  # One fixed order for every image

def shuffle_pixels(images):
    """Rearrange the 784 pixels of every image into the order perm."""
    return images.reshape(len(images), -1)[:, perm].reshape(images.shape)

In [ ]:
data_shuffled = d2l.FashionMNIST(batch_size=128)
data_shuffled.train, data_shuffled.val = [
    (shuffle_pixels(images), labels)
    for images, labels in (data_shuffled.train, data_shuffled.val)]

To us the shuffled images are noise.

In [ ]:
data_shuffled.visualize(next(iter(data_shuffled.val_dataloader())))

The MLP cannot tell the difference, and we need not train it again to know
this. Shuffling the pixels only reorders the inputs of its first layer, so
training on shuffled images is training on the original ones with the
columns of $\mathbf{W}^{(1)}$ reordered. Since every weight starts from the
same distribution, it is the same random experiment, and the MLP would again
reach about 87%. LeNet, however, now applies its kernels to $5\times5$
windows of pixels that were never neighbors. We train it with the same
settings as before and plot both validation curves.

In [ ]:
curves = {'pixels in place': trainer.history['val_acc']}
trainer.board.display = False
trainer.fit(LeNet(lr=2.0), data_shuffled)
curves['pixels shuffled'] = trainer.history['val_acc']
d2l.plot(list(range(1, trainer.max_epochs + 1)), list(curves.values()),
         'epoch', 'validation accuracy', legend=list(curves))
for name, acc in curves.items():  # A single epoch can land a point low
    print(f'{name}: {sum(acc[-5:]) / 5:.3f} averaged over the last 5 epochs')

On shuffled pixels LeNet ends several points lower than on the ordered
images, at about 85 or 86% in a typical run, which is at or a little below
the MLP. Its advantage is gone, and so is the doubt about where the
advantage came from: it was the advantage of knowing which pixels are
adjacent.

The plateau at chance is back as well, and it is as unpredictable as before.
On ordered pixels it lasted at most a few epochs. On shuffled pixels it
usually does the same, but once in a while it takes up most of the thirty
epochs, and such a run ends lower still. A reader who repeats the
experiment may well meet one. The plateau is a weakness of this network of
sigmoids and does not affect the comparison: in none of our runs, however
short its plateau, did LeNet on shuffled pixels end above the MLP.

One more observation ties LeNet back to the derivation. The network of
that section was to be equivariant in its layers and invariant in
its answer, and LeNet is neither. Its convolutions follow a shift only away
from the border, and its pooling layers absorb some one-pixel shifts and
pass on others (that section). Its head then flattens the
$5\times5$ maps and gives every position weights of its own, so nothing
makes the answer invariant. What tolerance to shifts LeNet has is partial,
and Fashion-MNIST, whose garments are all centered, never asks for more.

The two runs together show that the accuracy of the MLP can be had, and
slightly improved on, with fewer than a third of its parameters, and that
the improvement comes from the convolutions' knowledge of the grid. A kernel
learns a local pattern once and detects it at every position, whereas the
MLP must learn it anew wherever it occurs. On $28\times28$ images little
is at stake. On the megapixel photographs of that section, where a
single fully connected layer needs a billion weights or more, the same idea
decides whether the network can be built at all. Whether such a network can
also be trained is another matter, and the four sigmoids above give a first
hint of the difficulty.

## Summary

LeNet combines a convolutional encoder, in which the resolution shrinks while
the number of channels grows, with a small fully connected head. Its
convolutional layers hold few parameters and do most of the arithmetic; the
first fully connected layer holds most of the parameters. Trained like the
MLP of that section, it reaches about 88--90% validation
accuracy on Fashion-MNIST, against about 87% for the MLP, with fewer than a
third of its parameters. Its first layer learns silhouettes and vertical
edges from the labels alone. When the pixels are shuffled, the advantage
over the MLP disappears, so it came from the convolutions' knowledge of the
grid. The four sigmoids, each passing back at most $1/4$ of the gradient,
call for a much larger learning rate, 2.0 against 0.1, and they make the
start of training erratic.

The encoder followed by a classifier head remains the organization of most
convolutional networks, but nearly every component of LeNet has since been
replaced: the sigmoid by the ReLU, average pooling by max-pooling or strided
convolutions, and the dense head by global average pooling and a single
linear layer, with normalization layers added throughout. The next chapter
makes these changes one at a time, starting with AlexNet
(that section), and asks what each one contributes as networks grow
much deeper.

## Exercises

1. [code] **Modern components.** Retrain `LeNet` with the changes below,
   using the training settings of this section, and compare the
   validation accuracy of each variant after 30 epochs with that of
   `LeNet`.
    1. Replace average pooling with max-pooling.
    1. Replace the sigmoid activations with ReLU. If the loss diverges or
       the accuracy stays at chance, lower the learning rate until training
       succeeds, and report the learning rate you used.
    1. Make both changes together, and call the result the modernized
       LeNet. Which of the two changes accounts for most of the difference
       in accuracy?
    1. ● With the weights the model has at the start of training, measure
       for `LeNet` and for the modernized LeNet the norm of the gradient of
       the loss with respect to the first convolutional kernel, and the
       standard deviation of the logits across a minibatch. The derivation
       of Xavier initialization in that section assumes inputs
       with zero mean and no nonlinearity. Use the mean of the sigmoid's
       output and its slope
       $\sigma'(0) = 1/4$ to explain your measurements, the learning
       rate that `LeNet` needs, and the epochs at chance at the start of
       its training.

1. [code] **Original MNIST.** Write an `MNIST` data class by adapting
   `d2l.FashionMNIST`, and train both `LeNet` and the MLP of
   that section on it, with the settings that each
   uses for Fashion-MNIST. @LeCun.Bottou.Bengio.ea.1998 report a
   test error rate below 1% for LeNet-5 on MNIST.
    1. Compare the validation error rate of `LeNet` with this figure. The
       two are measured on the same images: as for Fashion-MNIST, our
       validation set is the dataset's official test set.
    1. For each dataset, compute the difference and the ratio of the
       validation error rates of the two networks. that section states
       that MNIST makes differences among model classes less visible. Does
       each of the two measures support this statement?

1. [code] **Deeper feature maps.** The text drew the feature maps of the
   first convolutional layer.
    1. Display the 16 feature maps of the second convolutional layer of
       the trained `LeNet`, after its sigmoid, for validation images of
       two classes that are easily confused, for example pullovers and
       coats. Can you still say what a channel responds to? Which channels
       differ most between the two classes?
    1. Train the modernized LeNet of problem
       1 and measure the maximum
       activation of each of its two convolutional layers, taken after the
       ReLU, on Fashion-MNIST validation images, on out-of-distribution
       photos such as a cat or a car, and on uniform random noise. Convert
       each photo to a $28 \times 28$ grayscale image, and keep every
       input in $[0, 1]$, the pixel range of `d2l.FashionMNIST`. Do the
       maxima for photos and noise fall inside or clearly outside the
       range observed on the validation images?

1. [code] **Overfitting sanity check.** Before trusting a full training
   run, check that a network can memorize a small dataset. Fix a subset
   of 50 Fashion-MNIST training images and train on it with all 50
   images in one minibatch, for at most 500 gradient steps.
    1. For both `LeNet` and the modernized LeNet of problem 1,
       record the number of steps until training accuracy exceeds 99%. If
       a network does not get there, diagnose which architectural or
       optimization choice is responsible.
    1. Name one bug that this check would catch and one that it would
       miss, and explain why the check cannot detect the second.

    *Adapted from Stanford CS231n,
    [Assignment 2](https://cs231n.github.io/assignments2024/assignment2/),
    Convolutional Networks notebook.*

1. [code] **Shifted garments.** The text argued that `LeNet` is only
   partly tolerant to shifts.
    1. Shift every validation image by one, two, and four pixels to the
       right, filling the vacated columns with zeros, and measure the
       validation accuracy of the trained `LeNet` on each shifted set.
       Which shifts does the network tolerate? Relate what you find to
       the two pooling layers and to the dense head.
    1. Explain why training `LeNet` on translated copies of its training
       images can improve the accuracy on shifted validation images even
       though its convolutional layers share weights across all
       locations.

    *Adapted from Michael Nielsen,
    [Neural Networks and Deep Learning](http://neuralnetworksanddeeplearning.com/chap6.html),
    Chapter 6.*

1. [code] **The dense head.** ● the table shows that the
   first fully connected layer holds 78% of LeNet's parameters.
    1. Replace the entire dense head, from the flatten through the output
       layer, by global average pooling followed by a single linear layer
       from 16 to 10 units. How many parameters does the network now
       have? Before training, predict whether the validation accuracy
       changes by more than one percentage point relative to `LeNet`; then
       retrain and compare.
    1. Instead connect the flattened 400-dimensional convolutional output
       directly to a ten-way linear output layer. How many parameters does
       this network have? Retrain, and compare the accuracy of the two
       reduced heads with each other and with your prediction. What does
       the comparison say about where LeNet's capacity is used?

    *Adapted from Michael Nielsen,
    [Neural Networks and Deep Learning](http://neuralnetworksanddeeplearning.com/chap6.html),
    Chapter 6.*

[Discussions](https://d2l.discourse.group/t/18000)